# EYES-DEFY-ANEMIA -- Phase 4 Classification -- new_way/ 3-fold retraining (kfold3)

Fixed-hyperparameter 3-fold cross-validation retraining of the **6 best combos** from the
`new_way/` 16-combo Optuna sweep (ranked by validation F1 in
`Output/version1/compare/new_way_model_comparison.xlsx`):

| Model | Tissue | Original F1 |
|---|---|---|
| ConvNeXt-Base | palpebral | 0.9333 |
| ConvNeXt-Large | palpebral | 0.9333 |
| CoAtNet-3 | palpebral | 0.8966 |
| EfficientNet-B3 | forniceal_palpebral | 0.8966 |
| MaxViT-Tiny | palpebral | 0.8750 |
| RegNetY-16GF | palpebral | 0.8667 |

**No Optuna in this notebook at all.** Each combo's `learning_rate`/`weight_decay`/`dropout_rate`
are fixed at its own already-found best trial's values (read from its real
`Output/version1/logs/*_study_summary.json` by `new_way/kfold/_generate_scripts.py`, not
hand-copied) -- the point of this run is a more robust *measurement* of each combo's real
performance (mean +/- std across 3 independent trainings), not a better hyperparameter search.

**Data:** each fold's TRAIN portion uses the offline, country-stratified label-balanced dataset
(`new_way/Offline_data_augmentation/`) *with* online augmentation (HorizontalFlip/Rotate) still
layered on top -- both, not one instead of the other, per explicit instruction. VAL and the
sealed TEST split are always real, unmodified, unaugmented images.

**Protocol:** 3-fold `StratifiedKFold` on the country+label compound key, over the pooled
train+val patients (test stays sealed) -- one fixed fold partition per tissue type, reused
identically across every model using that tissue type. `ReduceLROnPlateau` (factor=0.5,
patience=5, min_lr=1e-6), gradient clipping (max_norm=1.0), early stopping (patience=15, so the
scheduler gets a real chance to act first), 250-epoch ceiling, batch size 32 -- mirrors
`classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/cv_trainer_engine.py`'s
own fixed-hyperparameter CV numbers exactly (this project's own most directly analogous
precedent). Checkpoints saved fp16 (halves disk footprint across 6 architectures x 3 folds).

Real local timing (RTX 4050): EfficientNet-B3 ~0.94s/epoch, ConvNeXt-Large ~3.37s/epoch --
even a pessimistic worst case (all 18 fits run the full 250 epochs, no early stopping) is
~4.2h, comfortably one Kaggle session; realistically much less once early stopping engages.

`sync_outputs()` runs after every fold-completing training cell, so an interrupted session
still yields a downloadable zip of everything completed so far.

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

In [ ]:
# Diagnostic only -- kept for visibility in the saved run log.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

In [ ]:
# optuna is still required even though this notebook never runs a search --
# datapreparepipeline/trainer_engine.py (reused for ARCHITECTURE_REGISTRY/
# compute_metrics/evaluate) imports it unconditionally at module level.
# timm is required for coatnet_3 (not in torchvision at all).
!pip install -q optuna albumentations timm

## Data

Same Kaggle-dataset copy step as the original `classification-new-way.ipynb` -- verify
`SRC_DIR` against the `/kaggle/input` listing above before running if this is a fresh Kaggle
dataset attachment.

**`new_way/Offline_data_augmentation/` (the balanced train images + manifest.csv, ~4.3MB) is
NOT part of this copy step** -- it's small enough to live in the git repo directly and arrives
via the `git clone` above. The sanity-check cell below asserts it's actually present before any
training starts, so a forgotten `git push` fails loudly here instead of silently deep inside a
training run.

In [ ]:
import shutil
from pathlib import Path

# TODO: verify against the /kaggle/input listing cell above before running.
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

In [ ]:
# Confirms the offline-balanced dataset arrived via git clone (see markdown above) --
# fails loudly here, not deep inside the first training cell, if it wasn't committed/pushed.
import pandas as pd

manifest_path = Path("classification/new_way/Offline_data_augmentation/manifest.csv")
assert manifest_path.exists(), (
    f"{manifest_path} not found. Offline_data_augmentation/ must be committed and pushed "
    "to GitHub before this notebook can clone it -- it is NOT part of the Kaggle-uploaded "
    "processed-dataset."
)
manifest = pd.read_csv(manifest_path)
print(f"manifest.csv: {len(manifest)} rows")
print(manifest.groupby(["tissue_type", "country", "anemic_label"]).size())

## Registry + fold-building sanity check

Confirms the 6 target architectures actually build/forward-pass, and independently re-derives
each tissue type's 3-fold split (patient counts, no overlap between folds' validation sets) --
before any real training starts.

In [ ]:
import sys
sys.path.insert(0, "classification/datapreparepipeline")
sys.path.insert(0, "classification/new_way")
sys.path.insert(0, "classification/new_way/kfold")

import torch
from trainer_engine import ARCHITECTURE_REGISTRY, DEVICE

TARGET_ARCHS = ["efficientnet_b3", "maxvit_t", "regnet_y_16gf", "convnext_base", "coatnet_3", "convnext_large"]

for arch in TARGET_ARCHS:
    cfg = ARCHITECTURE_REGISTRY[arch]
    model = cfg["build_fn"](0.2).to(DEVICE)
    x = torch.randn(2, 3, cfg["input_size"], cfg["input_size"]).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    assert out.shape == (2, 1), f"{arch}: bad output shape {out.shape}"
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{arch:<20} OK  out={tuple(out.shape)}  trainable_params={n_trainable}")
    del model
print("\nAll 6 target architectures registered and working.")

In [ ]:
import kfold_engine as ke

for tissue in ["palpebral", "forniceal_palpebral"]:
    pool = ke.load_pool_df(tissue)
    folds = ke.build_folds(pool)
    print(f"=== {tissue}: pool={len(pool)} patients, {len(folds)} folds ===")
    total, seen_val = 0, set()
    for i, (tr, va) in enumerate(folds, 1):
        overlap = seen_val & set(va)
        assert not overlap, f"fold {i} overlaps a previous fold's val set: {overlap}"
        seen_val |= set(va)
        total += len(va)
        print(f"  fold {i}: train={len(tr)} val={len(va)}")
    assert total == len(pool), f"fold val sizes sum to {total}, expected {len(pool)}"
print("\nFold geometry OK for both tissue types.")

## Output syncing

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate new_way/Output/version2/{checkpoints,logs,plots}/ into a
    single top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/new_way_kfold3_results.zip. Called after EVERY training
    cell -- 6 models x 3 folds is a long unattended run, so whatever
    completed so far must always be downloadable."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/new_way/Output/version2") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/new_way_kfold3_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

## Training -- 6 combos x 3 folds = 18 fits, cheapest architecture first

Each script's `model_name` carries a `_kfold3` suffix so these results never collide with the
`version1` Optuna-sweep results under the un-suffixed name. Real local per-epoch timing
(RTX 4050): EfficientNet-B3 ~0.94s, ConvNeXt-Large ~3.37s -- ordering cheapest-first means an
interrupted session still banks the fastest, most numerous results first.

In [ ]:
# new_way kfold3 1/6 -- efficientnet_b3, forniceal_palpebral (lightest)
!python classification/new_way/kfold/train_kfold_efficientnet_b3_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way kfold3 2/6 -- maxvit_t, palpebral
!python classification/new_way/kfold/train_kfold_maxvit_t_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way kfold3 3/6 -- regnet_y_16gf, palpebral
!python classification/new_way/kfold/train_kfold_regnet_y_16gf_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way kfold3 4/6 -- convnext_base, palpebral
!python classification/new_way/kfold/train_kfold_convnext_base_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way kfold3 5/6 -- coatnet_3, palpebral
!python classification/new_way/kfold/train_kfold_coatnet_3_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way kfold3 6/6 -- convnext_large, palpebral (heaviest)
!python classification/new_way/kfold/train_kfold_convnext_large_palpebral_new_way.py
sync_outputs()

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever
combos completed) and zipped to `/kaggle/working/new_way_kfold3_results.zip`. Both are visible in
this notebook version's **Output** tab once you Save Version -> Save & Run All -- download the
zip directly from there, or browse the folder for individual files.

Each combo's `{model_name}_kfold_summary.json` has the `aggregate_across_folds` mean+/-std test
metrics -- that's the headline number this whole effort exists to produce.

In [ ]:
print("Final contents of /kaggle/working/outputs:")
for f in sorted(Path("/kaggle/working/outputs").rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to('/kaggle/working/outputs')}  ({f.stat().st_size / 1e6:.2f} MB)")

zip_path = Path("/kaggle/working/new_way_kfold3_results.zip")
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")